# 17.1 權重到底佔多少空間？


模型有兩千個參數，究竟需要兩千bytes，還是八千bytes？參數只表示可學數字的個數，還要知道每個數字用幾個位元保存。量化就是嘗試用較少的表示格子保存權重，節省模型儲存與讀取成本；開始談誤差以前，先把空間算清楚。前置是[矩陣暖身](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5)：一張64列、32欄的權重表有 `64×32=2048` 個數字。

bit（位元）可存兩種狀態，八個bits組成一個byte（位元組）。FP32用32bits存一個浮點數，每個占4bytes；FP16用16bits，每個2bytes；int8用8bits存一個整數，每個1byte。因此同一張2048格表，原始數字部分分別需要8192、4096與2048bytes。表的形狀沒變，改的是每格表示數字的方式，能表示的數值範圍與細節也會改變。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

weight = torch.zeros(64, 32)
for dtype in (torch.float32, torch.float16, torch.int8):
    converted = weight.to(dtype)
    count = converted.numel()
    per_value = converted.element_size()
    print(dtype, "元素", count, "每格bytes", per_value, "總bytes", count * per_value)

輸入全零只為隔離空間問題，不是訓練好的模型。`to(dtype)`建立指定數值型別的表；`numel`數元素，三次都應為2048；`element_size`回報每格實際占幾bytes，依序4、2、1。最後一欄對應上述8192、4096、2048。這段刻意不寫檔，因為檔案還可能含形狀、型別、名稱與容器資訊，單張表的數字bytes不等於完整檔案大小。

現在也有[完整已訓練模型的量化實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/quantization.json)：141,568個模型參數的FP32數值合計566,272 bytes，保存成`fp32.pt`則是588,358 bytes。4-bit轉換後參數個數仍相同，數值儲存168,736 bytes，`model.pt`為182,277 bytes；8-bit則是226,336與242,085 bytes。三個檔案都沒有optimizer，原始浮點檔仍含續訓相關的隨機狀態與不同metadata，因此檔案比不能直接當作純權重位元比。這是私有實跑產物的大小；另行公開、剔除隨機狀態後的匯出檔要重新量，不能照抄這些bytes。

4-bit／8-bit在這份實報中描述的是線性層加權係數的存法：115,200個係數換成低位元，其餘25,728個數字與640個偏置仍保存為FP32，共26,368個浮點數。偏置是每個輸出加權和後多加的數。量化版本還新增1,416個FP32刻度值，用來說明整數格子如何還原到原來的數值範圍；刻度的運算下一節再展開，本節先把它們的儲存也算進來。

| 數值儲存部分 | 原FP32模型（bytes） | 4-bit版本（bytes） | 8-bit版本（bytes） |
| --- | ---: | ---: | ---: |
| 115,200個加權係數 | 460,800 | 57,600 | 115,200 |
| 26,368個其餘數字與偏置，保留FP32 | 105,472 | 105,472 | 105,472 |
| 新增1,416個FP32還原刻度 | 0 | 5,664 | 5,664 |
| 合計 | 566,272 | 168,736 | 226,336 |

兩個4-bit係數放在一byte，所以115,200個需要115,200÷2＝57,600 bytes；保留的FP32數字占26,368×4＝105,472，刻度占1,416×4＝5,664。因此4-bit這一欄就是57,600＋105,472＋5,664＝168,736 bytes。這個合計已包括仍用FP32的數字與新增刻度；檔案的名稱、形狀與其他容器資訊，才是再加到合計以外的部分。因此要核對壓縮率，先問哪些數字改了表示，再按各部分的存法相加。

直接把任意浮點權重 `.to(torch.int8)`也不是完整量化方法：例如0.7會變成整數0，沒有記錄該如何還原的刻度。這裡全零在各型別都能準確表示，才方便只比較儲存。下一節會用scale說明少量整數格子如何對應原來的浮點範圍，才能合理討論壓縮誤差。

權重縮小也不代表整個執行記憶體同比例縮小。生成時還要保存[KV cache](https://birdhackor.github.io/tiny-perceptron-vlm/16.3.html)，向前計算有中間特徵；訓練時另外保存[權重梯度](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.6)，也就是損失對各權重的變化率，用來提示每個權重如何調整。優化器是執行更新的工具，還可能保存過去更新的歷史資訊，作為決定下一步大小的狀態。這些資料可能仍使用FP32或自己的型別。評估記憶體限制時，應分開記錄權重檔案、載入過程與執行的最高占用，不能只報這一張表。

練習只把行數64改成128，先預測元素變4096，各型別的總bytes全部加倍為16384、8192、4096，每格bytes保持不變；再執行核對。如果你能清楚說明哪些量由形狀決定、哪些由dtype決定，就準備好理解「參數少」與「低位元」兩種不同的省空間方式。
